In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

mesh_tagging_task_sp_26_dl_path = kagglehub.competition_download('mesh-tagging-task-sp-26-dl')

print('Data source import complete.')


100%|██████████| 71.2M/71.2M [00:00<00:00, 175MB/s]

Extracting files...


Data source import complete.


In [ ]:
!pip install -q transformers datasets scikit-learn pandas numpy tqdm torch accelerate
!pip install -q scipy sparse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 7.1 MB/s eta 0:00:00


In [ ]:
import ast
import re
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

# scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

# PyTorch / HuggingFace (for the BERT approach)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [ ]:
TRAIN_PATH = r'/content/training-set-100000.csv'
TEST_PATH  = r'/content/test-set-20000.csv'
JUDGE_PATH = r'/content/judge-set-10001-unannotated.csv'

# Load
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
judge_df = pd.read_csv(JUDGE_PATH)

print('Train shape :', train_df.shape)
print('Test  shape :', test_df.shape)
print('Judge shape :', judge_df.shape)

train_df.head(3)

Train shape : (100000, 4)
Test  shape : (20000, 4)
Judge shape : (10001, 3)


,pmid,title,abstractText,meshMajor
0,1372348,Endothelial cell activation in vasculitis of p...,To clarify the role of endothelial cells in th...,"['Aged', 'Autoantibodies', 'Biopsy', 'Cell Adh..."
1,17174061,The interleukin-8 (-251A/T) polymorphism is as...,AIMS: In light of recently found contribution ...,"['Adult', 'Aged', 'Aged, 80 and over', 'Carcin..."
2,544009,Identification of transformed rat liver epithe...,The growth potential of normal and transformed...,"['Animals', 'Cell Line', 'Cell Transformation,..."


In [ ]:
def parse_mesh(raw):
    """Convert string repr of a Python list to an actual list."""
    if pd.isna(raw):
        return []
    try:
        return ast.literal_eval(raw)
    except Exception:
        return []

train_df['labels'] = train_df['meshMajor'].apply(parse_mesh)
test_df['labels']  = test_df['meshMajor'].apply(parse_mesh)

# Verify
print(train_df['labels'].head(3).tolist())

[['Aged', 'Autoantibodies', 'Biopsy', 'Cell Adhesion Molecules', 'E-Selectin', 'Endothelium, Vascular', 'Female', 'Humans', 'Immunoenzyme Techniques', 'Intercellular Adhesion Molecule-1', 'Lymphocyte Function-Associated Antigen-1', 'Major Histocompatibility Complex', 'Male', 'Middle Aged', 'Muscles', 'Neutrophils', 'Peripheral Nerves', 'Vasculitis'], ['Adult', 'Aged', 'Aged, 80 and over', 'Carcinoma, Squamous Cell', 'Case-Control Studies', 'Female', 'Gene Frequency', 'Genetic Predisposition to Disease', 'Genotype', 'Germany', 'Greece', 'Humans', 'Interleukin-8', 'Male', 'Middle Aged', 'Mouth Neoplasms', 'Polymorphism, Genetic', 'Reverse Transcriptase Polymerase Chain Reaction', 'Risk'], ['Animals', 'Cell Line', 'Cell Transformation, Neoplastic', 'Cells, Cultured', 'Epithelial Cells', 'Liver', 'Liver Neoplasms, Experimental', 'Microscopy, Electron', 'Rats']]


In [ ]:
# Label counts per article
train_df['n_labels'] = train_df['labels'].apply(len)
print('Labels per article (train):')
print(train_df['n_labels'].describe())

# Most common MeSH terms
from collections import Counter
all_labels = [lbl for sublist in train_df['labels'] for lbl in sublist]
label_counts = Counter(all_labels)
print(f'\nTotal unique MeSH terms in training set: {len(label_counts)}')
print('Top 10 most common MeSH terms:')
for term, cnt in label_counts.most_common(10):
    print(f'  {term}: {cnt}')

Labels per article (train):
count    100000.000000
mean         13.056950
std           4.804108
min           1.000000
25%          10.000000
50%          13.000000
75%          16.000000
max          47.000000
Name: n_labels, dtype: float64

Total unique MeSH terms in training set: 22373
Top 10 most common MeSH terms:
  Humans: 61956
  Male: 37543
  Female: 37301
  Animals: 33144
  Adult: 22688
  Middle Aged: 20922
  Aged: 14980
  Mice: 10940
  Rats: 8500
  Adolescent: 8491


In [ ]:
mlb = MultiLabelBinarizer()
mlb.fit(train_df['labels'])          # fit only on train to avoid data leakage

Y_train = mlb.transform(train_df['labels'])
Y_test  = mlb.transform(test_df['labels'])

print('Number of labels (classes):', len(mlb.classes_))
print('Y_train shape:', Y_train.shape)
print('Y_test  shape:', Y_test.shape)

Number of labels (classes): 22373
Y_train shape: (100000, 22373)
Y_test  shape: (20000, 22373)


In [ ]:
def combine_text(row):
    title    = str(row['title'])        if pd.notna(row['title'])        else ''
    abstract = str(row['abstractText']) if pd.notna(row['abstractText']) else ''
    return (title + ' ' + abstract).strip()

train_df['text'] = train_df.apply(combine_text, axis=1)
test_df['text']  = test_df.apply(combine_text,  axis=1)
judge_df['text'] = judge_df.apply(combine_text, axis=1)

print('Sample text:\n', train_df['text'].iloc[0][:300])

Sample text:
 Endothelial cell activation in vasculitis of peripheral nerve and skeletal muscle. To clarify the role of endothelial cells in the pathogenesis of vasculitis affecting peripheral nerve and skeletal muscle, the endothelial expression of adhesion molecules and major histocompatibility antigens (MHC) i


## Approach A — TF-IDF + LinearSVC Baseline

In [ ]:
tfidf = TfidfVectorizer(
    max_features=100_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
)

X_train_tfidf = tfidf.fit_transform(train_df['text'])
X_test_tfidf  = tfidf.transform(test_df['text'])
X_judge_tfidf = tfidf.transform(judge_df['text'])

print('TF-IDF train matrix shape:', X_train_tfidf.shape)

TF-IDF train matrix shape: (100000, 100000)


In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

clf = OneVsRestClassifier(LinearSVC(C=0.5, max_iter=2000), n_jobs=-1)
clf.fit(X_train_tfidf, Y_train)
print('Training complete.')

Training complete.


In [ ]:
Y_pred_test = clf.predict(X_test_tfidf)

micro_f1 = f1_score(Y_test, Y_pred_test, average='micro', zero_division=0)
print(f'Test micro-F1: {micro_f1:.4f}')

Test micro-F1: 0.4103


In [ ]:
def make_submission(judge_df, Y_pred_binary, mlb, filename='submission_baseline.csv'):
    """Convert binary predictions back to MeSH label lists and save CSV."""
    predicted_labels = mlb.inverse_transform(Y_pred_binary)
    # Format as Python list string (e.g. "['Adult', 'Humans']")
    mesh_strings = [str(list(lbls)) for lbls in predicted_labels]
    submission = pd.DataFrame({
        'pmid': judge_df['pmid'].values,
        'meshMajor': mesh_strings
    })
    submission.to_csv(filename, index=False)
    print(f'Submission saved to {filename} — shape: {submission.shape}')
    return submission

Y_judge_pred = clf.predict(X_judge_tfidf)
sub_baseline = make_submission(judge_df, Y_judge_pred, mlb, 'submission_baseline.csv')
sub_baseline.head(5)

Submission saved to submission_baseline.csv — shape: (10001, 2)


,pmid,meshMajor
0,16854706,"['Adult', 'Female', 'Humans', 'Male']"
1,12943287,"['Aged', 'Female', 'Humans', 'Male', 'Tibia']"
2,9808709,"['Animals', 'Glutathione', 'JNK Mitogen-Activa..."
3,3856646,"['Adult', 'Female', 'Humans', 'Male', 'Mandibl..."
4,22033210,"['Animals', 'Cattle', 'Escherichia coli', 'Lip..."


In [ ]:
# ---- Hyperparameters ----
MODEL_NAME   = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
MAX_LEN      = 256          # tokens; increase to 512 if GPU allows
BATCH_SIZE   = 16           # reduce to 8 if OOM
EPOCHS       = 3
LR           = 2e-5
THRESHOLD    = 0.4          # sigmoid threshold for multi-label decision
NUM_LABELS   = len(mlb.classes_)

# Use only a subset for quick local testing; set to None to use full data on Kaggle GPU
TRAIN_LIMIT  = None         # e.g. 10000 for local dev, None for full run

print(f'NUM_LABELS : {NUM_LABELS}')
print(f'Device     : {DEVICE}')

NUM_LABELS : 22373
Device     : cuda


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MeSHDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=tokenizer, max_len=MAX_LEN):
        self.texts     = texts
        self.labels    = labels       # numpy binary array or None (judge set)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids'     : enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

print('Tokenizer ready.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [ ]:
class MeSHClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]   # [CLS] token
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits

model = MeSHClassifier(MODEL_NAME, NUM_LABELS).to(DEVICE)
print('Model parameters:', sum(p.numel() for p in model.parameters()) // 1_000_000, 'M')

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Model parameters: 126 M


In [ ]:
train_texts = train_df['text'].tolist()
test_texts  = test_df['text'].tolist()
judge_texts = judge_df['text'].tolist()

if TRAIN_LIMIT:
    train_texts = train_texts[:TRAIN_LIMIT]
    Y_train_use = Y_train[:TRAIN_LIMIT]
else:
    Y_train_use = Y_train

train_dataset = MeSHDataset(train_texts, Y_train_use)
test_dataset  = MeSHDataset(test_texts,  Y_test)
judge_dataset = MeSHDataset(judge_texts)            # no labels

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
judge_loader = DataLoader(judge_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

Train batches: 6250 | Test batches: 1250


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = total_steps // 10

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

criterion = nn.BCEWithLogitsLoss()
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')

Total steps: 18750 | Warmup steps: 1875


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


best_f1  = 0
for epoch in range(1, EPOCHS + 1):
    avg_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}')

Training: 100%|██████████| 6250/6250 [06:39<00:00, 15.65it/s]


Epoch 1/3  loss=0.0826


Training: 100%|██████████| 6250/6250 [06:38<00:00, 15.69it/s]


Epoch 2/3  loss=0.0037


Training: 100%|██████████| 6250/6250 [06:38<00:00, 15.68it/s]

Epoch 3/3  loss=0.0033


In [ ]:
@torch.no_grad()
def predict(model, loader, device, threshold=THRESHOLD):
    model.eval()
    all_preds = []
    for batch in tqdm(loader, desc='Predicting'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = model(input_ids, attention_mask)
        probs  = torch.sigmoid(logits).cpu().numpy()
        all_preds.append(probs)
    all_preds = np.vstack(all_preds)
    return (all_preds >= threshold).astype(int)


Y_pred_bert = predict(model, test_loader, DEVICE)
bert_micro_f1 = f1_score(Y_test, Y_pred_bert, average='micro', zero_division=0)
print(f'BERT Test micro-F1: {bert_micro_f1:.4f}')

Predicting: 100%|██████████| 1250/1250 [00:25<00:00, 49.47it/s]


BERT Test micro-F1: 0.2667


In [ ]:
Y_judge_bert = predict(model, judge_loader, DEVICE, threshold=THRESHOLD)
sub_bert = make_submission(judge_df, Y_judge_bert, mlb, 'submission_bert.csv')
sub_bert.head(5)

Predicting: 100%|██████████| 626/626 [00:13<00:00, 47.71it/s]


Submission saved to submission_bert.csv — shape: (10001, 2)


,pmid,meshMajor
0,16854706,"['Adult', 'Female', 'Humans', 'Male']"
1,12943287,"['Adult', 'Female', 'Humans', 'Male', 'Middle ..."
2,9808709,"['Animals', 'Mice', 'Rats']"
3,3856646,"['Female', 'Humans', 'Male']"
4,22033210,"['Animals', 'Mice']"


In [ ]:
# ---- Format check ----
sub = pd.read_csv('submission_bert.csv')

assert list(sub.columns) == ['pmid', 'meshMajor'], 'Wrong columns!'
assert len(sub) == len(judge_df), f'Row count mismatch: {len(sub)} vs {len(judge_df)}'

# Check meshMajor is case-sensitive string list repr
sample_labels = parse_mesh(sub['meshMajor'].iloc[0])
print('Sample predictions for first article:')
print(sample_labels)

# Check all pmids are present
missing = set(judge_df['pmid']) - set(sub['pmid'])
print(f'Missing pmids: {len(missing)}')
print('\n Submission looks good!' if not missing else 'Missing pmids!')

Sample predictions for first article:
['Adult', 'Female', 'Humans', 'Male']
Missing pmids: 0

 Submission looks good!


In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'mlb_classes'     : mlb.classes_,
    'threshold'       : THRESHOLD,
}, 'mesh_model_checkpoint.pt')
print('Checkpoint saved.')

Checkpoint saved.


In [1]:
pip freeze > requirements.txt